# Stage 5 validation

In [1]:
import math
from datetime import date, timedelta

from backend.data_engine.contracts import SyntheticCompanyConfig
from backend.data_engine.ingest.base import (
    MacroObservation, MacroSeries, MarketObservation, MarketSeries,
    SourceMetadata, SourceStatus,
)
from backend.data_engine.ingest.synthetic import generate_company
from backend.data_engine.risk_inputs import build_market_inputs
from backend.risk_engine.engine import QuantitativeRiskEngine


def closes(count, drift, start=100.0):
    out = [start]
    for i in range(1, count):
        out.append(out[-1] * math.exp(drift if i % 2 == 0 else -drift))
    return out


def mseries(symbol, data, start=date(2026, 1, 1)):
    return MarketSeries(
        symbol=symbol,
        observations=[
            MarketObservation(date=start + timedelta(days=i), close=c)
            for i, c in enumerate(data)
        ],
        metadata=SourceMetadata(
            source='nb-fixture', url='notebook://stage5',
            license_tag='synthetic', cache_status='test',
            status=SourceStatus.FETCHED,
        ),
    )


def macro(series_id, values):
    return MacroSeries(
        series_id=series_id,
        observations=[
            MacroObservation(
                date=date(2021, 1, 1) + timedelta(days=30 * i), value=v)
            for i, v in enumerate(values)
        ],
        metadata=SourceMetadata(
            source='nb-fixture', url='notebook://stage5',
            license_tag='synthetic', cache_status='test',
            status=SourceStatus.FETCHED,
        ),
    )


dataset = generate_company(SyntheticCompanyConfig(seed=1001, periods=24))
engine = QuantitativeRiskEngine()
by_id = lambda r: {m.metric_id: m for m in r.metrics}
print('dataset:', dataset.profile.name)


dataset: Greystone Industries


In [2]:
inputs = build_market_inputs(
    equity=mseries('EQUITY', closes(150, 0.02)),
    benchmark=mseries('INDEX', closes(150, 0.005)),
    fx=mseries('DEXINUS', closes(150, 0.001)),
)
m = by_id(engine.evaluate_company(dataset, market_inputs=inputs))
print('beta:', round(m['beta'].value, 4), m['beta'].status)
print('legs:', m['beta'].inputs_used.get('asset_symbol'),
      '/', m['beta'].inputs_used.get('benchmark_symbol'))
print('equity_vol:', round(m['equity_volatility'].value, 3))
print('fx_vol:', round(m['fx_volatility'].value, 3))
assert m['beta'].value != 1.0

shocked = build_market_inputs(
    equity=mseries('EQUITY', closes(150, 0.02)),
    benchmark=mseries('INDEX', closes(150, 0.005)),
    fx=mseries('DEXINUS', closes(150, 0.05)),
)
m2 = by_id(engine.evaluate_company(dataset, market_inputs=shocked))
assert m2['fx_volatility'].value != m['fx_volatility'].value
assert m2['equity_volatility'].value == m['equity_volatility'].value
assert m2['var_95'].value == m['var_95'].value
print('separation OK')


beta: 4.0 valid
legs: EQUITY / INDEX
equity_vol: 31.748
fx_vol: 1.587
separation OK


In [3]:
periods = [
    {'revenue': 1000.0, 'gross_profit': 400.0 - i * (20.0 / 23)}
    for i in range(24)
]
mi = build_market_inputs(
    cpi=macro('CPIAUCSL', [300.0 + i for i in range(24)]),
    fedfunds=macro('FEDFUNDS', [2.0 + 0.05 * i for i in range(60)]),
    periods=periods,
)
mm = by_id(engine.evaluate_company(dataset, market_inputs=mi))
print('infl:', round(mm['inflation_passthrough'].value, 4))
print('rate:', round(mm['rate_environment'].value, 4))

empty = by_id(engine.evaluate_company(dataset))
print('beta w/o inputs:', empty['beta'].status)
print('infl w/o inputs:', empty['inflation_passthrough'].status)
assert str(empty['beta'].status) == 'unavailable'

a = engine.evaluate_company(dataset, market_inputs=inputs).composite.score
b = engine.evaluate_company(dataset, market_inputs=inputs).composite.score
assert a == b
print('deterministic:', round(a, 4))
print('Stage 5 notebook OK')


infl: -9.6667
rate: 1.475
beta w/o inputs: unavailable
infl w/o inputs: missing_input


deterministic: 38.8037
Stage 5 notebook OK
